In [4]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [5]:
import polars as pl
from polars import selectors as cs, String
from extract import endpoints, get, get_leagues
from transform import transform_sports, transform_leagues, transform_seasons, transform_divisions, transform_teams, \
    TeamsBySeasonSchema
from transform import DivisionSchema, DivisionSeasonsSchema,SportSchema, LeagueSchema, SeasonSchema, TeamSchema, TeamsBySeasonSchema
from transform import SportsSeasons, LeagueCollection
from pathlib import Path

In [6]:
data = Path('data/statsapi')
tables = endpoints.keys()
paths = [data/(table+'.parquet') for table in tables]

In [7]:
for table, path in zip(tables, paths):
    if not path.exists():
        get(table).collect().write_parquet(path)

In [8]:
sports = pl.scan_parquet(data/'sports.parquet')
leagues = pl.scan_parquet(data/'leagues.parquet')
divisions = pl.scan_parquet(data/'divisions.parquet')
seasons = pl.scan_parquet(data/'seasons.parquet')
teams = pl.scan_parquet(data/'teams.parquet')

In [9]:
sports = transform_sports(sports)
sports = SportSchema.validate(sports, cast=True).lazy()

In [10]:
leagues = transform_leagues(leagues)
leagues = LeagueSchema.validate(leagues, cast=True).lazy()

In [11]:
seasons  = transform_seasons(seasons)
seasons = SeasonSchema.validate(seasons, cast=True).lazy()

In [12]:
SC, bad = SportsSeasons.filter(
    {
        'sports': sports,
        'seasons': seasons,
    }
)

In [13]:
bad['sports']._df

sport_id,sport_code,sport_name,sport_abbr,sort_order,sport_link,primary_key,sport_id|nullability,sport_code|nullability,sport_code|unique,sport_code|max_length,sport_name|nullability,sport_name|unique,sport_abbr|nullability,sport_abbr|unique,sort_order|nullability,sort_order|unique,sport_link|nullability,sport_link|unique,sport_link|regex,seasons_sports
u32,str,str,str,u32,str,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool
21,"""min""","""Minor League Baseball""","""Minors""",1402,"""/api/v1/sports/21""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,false
61,"""nlb""","""Negro League Baseball""","""NLB""",2401,"""/api/v1/sports/61""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,false
32,"""kor""","""Korean Baseball Organization""","""KOR""",2601,"""/api/v1/sports/32""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,false
31,"""jml""","""Nippon Professional Baseball""","""NPB""",2701,"""/api/v1/sports/31""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,false
509,"""nae""","""International Baseball (18U)""","""18U""",3503,"""/api/v1/sports/509""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,false
510,"""nas""","""International Baseball (16 and…","""16U""",3505,"""/api/v1/sports/510""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,false
6005,"""ame""","""International Baseball (amateu…","""AME""",3509,"""/api/v1/sports/6005""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,false
52,"""oly""","""Olympic Baseball""","""OLY""",3511,"""/api/v1/sports/52""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,false


In [14]:
div, div_seasons = transform_divisions(divisions, leagues)
divisions = DivisionSchema.validate(div, cast=True).lazy()
division_seasons = DivisionSeasonsSchema.validate(div_seasons, cast=True).lazy()

In [15]:
teams, team_seasons = transform_teams(teams)
teams = TeamSchema.validate(teams, cast=True).lazy()
team_seasons = TeamsBySeasonSchema.validate(team_seasons, cast=True).lazy()

In [16]:
LC = LeagueCollection.validate(
    {
        'leagues': leagues,
        'divisions': divisions,
        'seasons': seasons,
        'team_seasons': team_seasons
    }
)